In [1]:
import pandas as pd
import os
from pathlib import Path

best_model = True
experiment_logs_path = Path(r"logs\AllScenarios")

main_results_df = pd.read_csv(experiment_logs_path / "main_results.csv")

main_results_df

,Architecture,Encoder,Pretrained,Parameter Count,Inference Time,Last Validation Dice,Last Validation IoU
0,unet,efficientnet-b0,scratch-small,6250893,8.459188,0.941616,0.897465
1,unet,efficientnet-b0,scratch-large,6250893,8.429700,0.985323,0.971434
2,unet,efficientnet-b0,other-dataset,6250893,8.664557,0.962139,0.930684
3,unet,efficientnet-b0,same-dataset,6250893,8.470039,0.955302,0.920157
4,unet,efficientnet-b1,scratch-small,8756529,11.392201,0.948036,0.910084
...,...,...,...,...,...,...,...
387,unet++,resnet18,same-dataset,15964177,7.081173,0.960436,0.928583
388,unet++,resnet34,scratch-small,26072337,8.675890,0.904793,0.847414
389,unet++,resnet34,scratch-large,26072337,8.733483,0.982674,0.966825
390,unet++,resnet34,other-dataset,26072337,8.502233,0.967227,0.938512


In [2]:
# trained_models_path = Path(r"\\zvsl.mvl6.uni-tuebingen.de\Employees\06_Datasets\Bjoern\Segmentation\trained_models\Experiment2and3\architecture_models")
trained_models_path = Path(r"D:\Bjoern\Segmentation\trained_models\AllScenarios\architecture_models")

for _, row in main_results_df.iterrows():
    model_name = f"{row['Architecture']}_{row['Encoder']}_zscore_{row['Pretrained']}_final_model.pth"

    model_path = trained_models_path / model_name

    if not model_path.exists():
        print(f"Model {model_name} does not exist at {model_path}")

In [3]:
import json
import dataset

with open(experiment_logs_path / "participant_split.json", "r") as f:
    participant_split = json.load(f)


# path_to_dataset = Path(r"F:\Python\SAM2\OCTDatasetOIMHS")
path_to_dataset = Path(r"D:\Bjoern\Segmentation\datasets\OCTDatasetOIMHS")

test_dataset = dataset.OIMHSDataset(
    participants = participant_split["test"],
    path = path_to_dataset,
    augment = False,
    max_rotate_deg = 0,
    return_numpy = False,
    normalize = "zscore"
)

print(f"Number of test samples: {len(test_dataset)}")

Number of test samples: 1243


In [4]:
import torch

def calc_metrics(logits, mask):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

    preds_f = preds.view(1, -1)
    masks_f = mask.view(1, -1)

    intersection = (preds_f * masks_f).sum()
    pred_sum = preds_f.sum()
    mask_sum = masks_f.sum()

    union = pred_sum + mask_sum - intersection

    dice_score = (2 * intersection) / (pred_sum + mask_sum)
    iou_score = intersection / union

    return dice_score.item(), iou_score.item()

In [5]:
row

Architecture                  unet++
Encoder                     resnet34
Pretrained              same-dataset
Parameter Count             26072337
Inference Time              8.627909
Last Validation Dice         0.96122
Last Validation IoU         0.929508
Name: 391, dtype: object

In [6]:

# if best_model:
#     out_file_path = experiment_logs_path / "metrics_per_image_best_model.csv"
# else:
#     out_file_path = experiment_logs_path / "metrics_per_image.csv"

# # out_file_path = experiment_logs_path / "metrics_per_image.csv"

# def read_combinations(file_path):
#     combinations = set()
#     with open(file_path, 'r') as f:
#         for line in f:
#             line = line.strip()
#             if line and not line.startswith('#'):
#                 parts = line.split(',')
#                 # print(parts)
#                 if len(parts) >= 3:
#                     architecture, encoder, pretrained = parts[:3]
#                     combinations.add((architecture.strip(), encoder.strip(), pretrained.strip()))
#     return combinations

# if out_file_path.exists():
#     existing_combinations = read_combinations(out_file_path)
# else:
#     existing_combinations = set()
# print(existing_combinations)

In [7]:
import torch
import segmentation_models_pytorch as smp
from tqdm import tqdm

i = 0
use_gpu = True

device = torch.device("cuda" if torch.cuda.is_available() and use_gpu else "cpu")

if best_model:
    out_file_path = experiment_logs_path / "metrics_per_image_best_model.csv"
else:
    out_file_path = experiment_logs_path / "metrics_per_image.csv"



# # out_file_path = experiment_logs_path / "metrics_per_image.csv"

def read_combinations(file_path):
    combinations = set()
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                parts = line.split(',')
                if len(parts) >= 3:
                    architecture, encoder, pretrained = parts[:3]
                    combinations.add((architecture.strip(), encoder.strip(), pretrained.strip()))
    return combinations

if out_file_path.exists():
    combinations = read_combinations(out_file_path)
    writer = open(out_file_path, "a")
else:
    combinations = set()
    writer = open(out_file_path, "w")
    header = "Architecture,Encoder,Pretrained,Participant,File,Dice,IoU\n"
    writer.write(header)

n_models = len(main_results_df)
current_model_n = 0

alias = {
        "unet": "unet",
        "u-net": "unet",
        "unet++": "unetplusplus",
        "unetplusplus": "unetplusplus",
        "deeplabv3+": "deeplabv3plus",
        "deeplabv3plus": "deeplabv3plus",
        "deeplab": "deeplabv3plus",
        "segformer": "segformer",
        "fpn": "fpn",
        "pspnet": "pspnet",
        "linknet": "linknet",
        "pan": "pan",
        "manet": "manet",
        "upernet": "upernet",
        "dpt": "dpt",
    }

for _, row in main_results_df.iterrows():
    current_model_n += 1
    model_type = "best_model" if best_model else "final_model"
    model_name = f"{row['Architecture']}_{row['Encoder']}_zscore_{row['Pretrained']}_{model_type}.pth"

    if (row['Architecture'], row['Encoder'], row['Pretrained']) in combinations:
        print(f"Skipping model {model_name} as it has already been processed.")
        continue

    model_path = trained_models_path / model_name

    if not model_path.exists():
            print(f"Model {model_name} does not exist at {model_path}")
            continue
    
    model = smp.create_model(
        arch=alias.get(row['Architecture'], row['Architecture']),
        encoder_name=row['Encoder'],
        in_channels=1,
        classes=1,
        activation=None
    )

    
    model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))


    model.to(device)
    model.eval()
    with torch.no_grad():
        for i, (img, mask) in tqdm(enumerate(test_dataset), total=len(test_dataset), desc=f"Model {current_model_n}/{n_models}: Architecture: {row['Architecture']}, Encoder: {row['Encoder']}, Pretrained: {row['Pretrained']}"):
            img_path, _ = test_dataset.samples[i]
            img = img.to(device)
            mask = mask.to(device)
            logits = model(img.unsqueeze(0))
            dice_score, iou_score = calc_metrics(logits, mask)

            folder = img_path.parent.name
            file_name = img_path.name

            # print(f"Sample {i}: Folder: {folder}, File name: {file_name}, Dice score: {dice_score:.4f}, IoU score: {iou_score:.4f}")

            writer.write(f"{row['Architecture']},{row['Encoder']},{row['Pretrained']},{folder},{file_name},{dice_score},{iou_score}\n")
            writer.flush()

writer.close()
    

d:\Bjoern\Segmentation\pytorch212\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Model 1/392: Architecture: unet, Encoder: efficientnet-b0, Pretrained: scratch-small: 100%|██████████| 1243/1243 [00:37<00:00, 33.20it/s]
Model 2/392: Architecture: unet, Encoder: efficientnet-b0, Pretrained: scratch-large: 100%|██████████| 1243/1243 [00:23<00:00, 52.09it/s]
Model 3/392: Architecture: unet, Encoder: efficientnet-b0, Pretrained: other-dataset: 100%|██████████| 1243/1243 [00:23<00:00, 53.50it/s]
Model 4/392: Architecture: unet, Encoder: efficientnet-b0, Pretrained: same-dataset: 100%|██████████| 1243/1243 [00:22<00:00, 54.78it/s]
Model 5/392: Architecture: unet, Encoder: efficientnet-b1, Pretrained: scratch-small: 100%|██████████| 1243/1243 [00:27<00:00, 45.46it/s]
Model 6/392: Architecture: unet, Encoder: efficientnet